# M0 — аудит данных и контракт валидации

**Цель.** Проверить поставку данных, структуру positive query–item пар, пересечения с benchmark-корпусом и построить воспроизводимый локальный протокол до первого retrieval-эксперимента.

**Критерий готовности M0.** Ноутбук должен:

- подтвердить схему, количество строк, пропуски и целостность идентификаторов;
- описать логические запросы и число известных positive items на запрос;
- измерить пересечения train с benchmark items/queries;
- зафиксировать group-disjoint validation proxy;
- содержать точную реализацию macro Recall@50 и валидатор `answer.csv`.

> Этот notebook выполняет только аудит. BM25 и другие retriever-ы начинаются на M1.


## План ноутбука

1. **Поставка и схема** — проверяем, что три Parquet-файла читаются и имеют ожидаемую структуру.
2. **Идентификаторы и labels** — определяем логические запросы, positive items и возможные дубликаты.
3. **Пересечения train и benchmark** — оцениваем, какие train-позитивы и query-history применимы к финальному corpus.
4. **Текст и структура corpus** — смотрим поля, категории и их практическую ценность для retrieval.
5. **Validation proxy** — строим group-disjoint split, на котором будут сравниваться следующие эксперименты.
6. **Метрика и submission** — проверяем macro Recall@50 и формат будущего `answer.csv`.

Каждый следующий раздел содержит только необходимые ячейки для указанного шага; M0 не обучает и не сравнивает retrieval-модели.

In [1]:
from __future__ import annotations

import os
import platform
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

REPO_ROOT = Path.cwd()
DATA_DIR = Path(os.environ.get("AVITO_DATA_DIR", "/Users/kite/Downloads/dataset"))
TRAIN_PATH = DATA_DIR / "train.parquet"
BENCHMARK_QUERIES_PATH = DATA_DIR / "benchmark_queries.parquet"
BENCHMARK_ITEMS_PATH = DATA_DIR / "benchmark_items.parquet"

SEARCH_COLUMNS = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]
ITEM_TEXT_COLUMNS = [
    "item_title_raw",
    "item_infm_params_text",
    "item_description_raw",
]
EXPECTED_FILES = [TRAIN_PATH, BENCHMARK_QUERIES_PATH, BENCHMARK_ITEMS_PATH]

assert all(path.exists() for path in EXPECTED_FILES), (
    "Dataset not found. Set AVITO_DATA_DIR or place the three Parquet files in data/."
)

print(f"Repository: {REPO_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Python: {platform.python_version()} | pandas: {pd.__version__} | pyarrow: {pa.__version__}")


Repository: /Users/kite/Documents/Карьера/avito-services-candidate-generation
Data directory: /Users/kite/Downloads/dataset
Python: 3.14.3 | pandas: 3.0.5 | pyarrow: 25.0.1


## 1. Поставка и схема

Сначала считываем только Parquet metadata. Это дешёвая проверка row count,
типов и количества row groups без декодирования полного текста объявлений.


In [2]:
def parquet_catalog(paths):
    rows = []
    for path in paths:
        parquet = pq.ParquetFile(path)
        rows.append(
            {
                "file": path.name,
                "rows": parquet.metadata.num_rows,
                "columns": len(parquet.schema_arrow),
                "row_groups": parquet.metadata.num_row_groups,
                "disk_mb": round(path.stat().st_size / 2**20, 2),
                "schema": ", ".join(
                    f"{field.name}: {field.type}" for field in parquet.schema_arrow
                ),
            }
        )
    return pd.DataFrame(rows)

catalog = parquet_catalog(EXPECTED_FILES)
display(catalog)


def null_profile(path, batch_size=65_536):
    parquet = pq.ParquetFile(path)
    null_counts = {name: 0 for name in parquet.schema_arrow.names}
    for batch in parquet.iter_batches(batch_size=batch_size):
        for name, column in zip(batch.schema.names, batch.columns):
            null_counts[name] += column.null_count
    total_rows = parquet.metadata.num_rows
    return pd.DataFrame(
        {
            "column": parquet.schema_arrow.names,
            "dtype": [str(parquet.schema_arrow.field(name).type) for name in parquet.schema_arrow.names],
            "null_count": [null_counts[name] for name in parquet.schema_arrow.names],
        }
    ).assign(null_rate=lambda frame: frame["null_count"] / total_rows)

null_profiles = {
    "train": null_profile(TRAIN_PATH),
    "benchmark_queries": null_profile(BENCHMARK_QUERIES_PATH),
    "benchmark_items": null_profile(BENCHMARK_ITEMS_PATH),
}
for name, profile in null_profiles.items():
    print(f"\n{name}")
    display(profile.sort_values(["null_rate", "column"], ascending=[False, True]).reset_index(drop=True))


,file,rows,columns,row_groups,disk_mb,schema
0,train.parquet,497673,19,1,467.53,"search_query: string, search_location_id: int6..."
1,benchmark_queries.parquet,2452,6,1,0.12,"query_id: large_string, search_query: large_st..."
2,benchmark_items.parquet,189212,14,1,186.61,"item_title_raw: string, item_rating_reviews_co..."



train


,column,dtype,null_count,null_rate
0,item_rating,double,28232,0.056728
1,item_rating_reviews_count,double,19211,0.038602
2,item_latitude,"decimal128(17, 15)",4,0.000008
3,item_longitude,"decimal128(18, 15)",4,0.000008
4,item_category_id,int64,0,0.000000
5,item_description_raw,string,0,0.000000
6,item_id,string,0,0.000000
7,item_infm_params_text,string,0,0.000000
8,item_is_message_forbidden,bool,0,0.000000
9,item_is_phone_hidden,bool,0,0.000000



benchmark_queries


,column,dtype,null_count,null_rate
0,query_id,large_string,0,0.0
1,search_category,int64,0,0.0
2,search_infm_params_text,large_string,0,0.0
3,search_is_delivery_search,int32,0,0.0
4,search_location_id,int64,0,0.0
5,search_query,large_string,0,0.0



benchmark_items


,column,dtype,null_count,null_rate
0,item_rating,double,17631,0.093181
1,item_rating_reviews_count,double,11615,0.061386
2,item_description_raw,string,35,0.000185
3,item_latitude,"decimal128(17, 15)",1,0.000005
4,item_longitude,"decimal128(18, 15)",1,0.000005
5,item_category_id,int64,0,0.000000
6,item_id,string,0,0.000000
7,item_infm_params_text,string,0,0.000000
8,item_is_message_forbidden,bool,0,0.000000
9,item_is_phone_hidden,bool,0,0.000000


## 2. Идентификаторы и базовая целостность

Проверяем уникальность `query_id`, уникальность benchmark `item_id`, формат
16-символьных hexadecimal item IDs и допустимость train positives в будущем
benchmark-корпусе.


In [3]:
HEX16 = re.compile(r"^[0-9a-f]{16}$")


def id_quality(series, require_hex):
    values = series.dropna().astype("string")
    invalid_format = int((~values.str.fullmatch(HEX16.pattern)).sum()) if require_hex else 0
    return {
        "rows": len(series),
        "nulls": int(series.isna().sum()),
        "unique": int(values.nunique()),
        "duplicates": int(values.duplicated().sum()),
        "invalid_format": invalid_format,
    }

train_item_ids = pd.read_parquet(TRAIN_PATH, columns=["item_id"])["item_id"]
benchmark_item_ids = pd.read_parquet(BENCHMARK_ITEMS_PATH, columns=["item_id"])["item_id"]
benchmark_query_ids = pd.read_parquet(BENCHMARK_QUERIES_PATH, columns=["query_id"])["query_id"]

id_audit = pd.DataFrame(
    [
        {"entity": "train.item_id", **id_quality(train_item_ids, require_hex=True)},
        {"entity": "benchmark_items.item_id", **id_quality(benchmark_item_ids, require_hex=True)},
        {"entity": "benchmark_queries.query_id", **id_quality(benchmark_query_ids, require_hex=False)},
    ]
)
display(id_audit)

benchmark_item_id_set = set(benchmark_item_ids.dropna().astype(str))
train_item_id_set = set(train_item_ids.dropna().astype(str))
item_overlap = train_item_id_set & benchmark_item_id_set
print(
    f"Train unique item IDs in benchmark corpus: {len(item_overlap):,} / "
    f"{len(train_item_id_set):,} ({len(item_overlap) / len(train_item_id_set):.2%})"
)


,entity,rows,nulls,unique,duplicates,invalid_format
0,train.item_id,497673,0,344825,152848,0
1,benchmark_items.item_id,189212,0,189212,0,0
2,benchmark_queries.query_id,2452,0,2452,0,0


Train unique item IDs in benchmark corpus: 18,142 / 344,825 (5.26%)


## 3. Логические запросы и labels

Каждая строка `train` — positive pair, а не независимый запрос. Логический
запрос определяется всеми `search_*` полями. Мы кодируем одинаковые наборы
признаков одним group ID и агрегируем все known positives.


In [4]:
train_pairs = pd.read_parquet(
    TRAIN_PATH,
    columns=[*SEARCH_COLUMNS, "item_id", "item_category_id", "item_location_id"],
)
benchmark_queries = pd.read_parquet(BENCHMARK_QUERIES_PATH)


def canonical_query_frame(frame):
    result = frame[SEARCH_COLUMNS].copy()
    for column in ["search_query", "search_infm_params_text"]:
        result[column] = (
            result[column]
            .astype("string")
            .fillna("<NA>")
            .str.lower()
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )
    for column in ["search_location_id", "search_is_delivery_search", "search_category"]:
        result[column] = result[column].astype("string").fillna("<NA>")
    return result

all_query_contexts = pd.concat(
    [canonical_query_frame(train_pairs), canonical_query_frame(benchmark_queries)],
    ignore_index=True,
)
all_group_ids, _ = pd.factorize(pd.MultiIndex.from_frame(all_query_contexts), sort=False)
train_pairs["query_group"] = all_group_ids[: len(train_pairs)]
benchmark_queries["query_group"] = all_group_ids[len(train_pairs) :]

query_labels = (
    train_pairs.groupby("query_group", sort=False)
    .agg(n_train_pairs=("item_id", "size"), n_known_positives=("item_id", "nunique"))
    .reset_index()
)

query_summary = pd.DataFrame(
    {
        "train_positive_pairs": [len(train_pairs)],
        "logical_train_queries": [len(query_labels)],
        "duplicate_train_pairs": [int(train_pairs.duplicated(["query_group", "item_id"]).sum())],
        "median_known_positives_per_query": [query_labels["n_known_positives"].median()],
        "p95_known_positives_per_query": [query_labels["n_known_positives"].quantile(0.95)],
        "max_known_positives_per_query": [query_labels["n_known_positives"].max()],
        "benchmark_query_rows": [len(benchmark_queries)],
        "unique_benchmark_query_contexts": [benchmark_queries["query_group"].nunique()],
        "duplicate_benchmark_query_context_rows": [int(benchmark_queries.duplicated("query_group").sum())],
    }
)
display(query_summary.T.rename(columns={0: "value"}))

distribution = (
    query_labels["n_known_positives"]
    .value_counts()
    .sort_index()
    .rename_axis("known_positives")
    .reset_index(name="logical_queries")
)
display(distribution.head(12))


,value
train_positive_pairs,497673.0
logical_train_queries,354313.0
duplicate_train_pairs,30644.0
median_known_positives_per_query,1.0
p95_known_positives_per_query,3.0
max_known_positives_per_query,278.0
benchmark_query_rows,2452.0
unique_benchmark_query_contexts,2452.0
duplicate_benchmark_query_context_rows,0.0


,known_positives,logical_queries
0,1,301082
1,2,33661
2,3,9362
3,4,3925
4,5,2023
5,6,1187
6,7,739
7,8,542
8,9,388
9,10,255


In [5]:
train_groups = set(train_pairs["query_group"])
benchmark_groups = set(benchmark_queries["query_group"])
exact_query_overlap = train_groups & benchmark_groups

benchmark_overlap_mask = train_pairs["item_id"].astype(str).isin(benchmark_item_id_set)
proxy_pairs = train_pairs.loc[benchmark_overlap_mask].copy()
proxy_labels = (
    proxy_pairs.groupby("query_group", sort=False)["item_id"]
    .nunique()
    .rename("n_proxy_positives")
)

alignment_audit = pd.DataFrame(
    {
        "metric": [
            "benchmark query contexts with an exact train context",
            "exact train contexts represented in benchmark queries",
            "train positive rows whose item exists in benchmark corpus",
            "logical train queries with >=1 benchmark-corpus positive",
            "unique train positives represented in benchmark corpus",
            "positive pairs with same search/item location ID",
            "positive pairs with same search/item category ID",
        ],
        "value": [
            len(benchmark_groups & train_groups),
            len(exact_query_overlap),
            int(benchmark_overlap_mask.sum()),
            int(proxy_labels.size),
            int(proxy_pairs["item_id"].nunique()),
            float((train_pairs["search_location_id"] == train_pairs["item_location_id"]).mean()),
            float((train_pairs["search_category"] == train_pairs["item_category_id"]).mean()),
        ],
    }
)
display(alignment_audit)
print("Proxy labels are known train positives that are retrievable from benchmark_items.")
print(
    f"Proxy label-cardinality median={proxy_labels.median():.0f}, "
    f"p95={proxy_labels.quantile(0.95):.0f}, max={proxy_labels.max():.0f}"
)


,metric,value
0,benchmark query contexts with an exact train c...,108.000000
1,exact train contexts represented in benchmark ...,108.000000
2,train positive rows whose item exists in bench...,33010.000000
3,logical train queries with >=1 benchmark-corpu...,26549.000000
4,unique train positives represented in benchmar...,18142.000000
5,positive pairs with same search/item location ID,0.831020
6,positive pairs with same search/item category ID,0.999894


Proxy labels are known train positives that are retrievable from benchmark_items.
Proxy label-cardinality median=1, p95=2, max=25


### 3.1. Relaxed query-history overlap

После строгого full-context overlap следующие ячейки измеряют более широкое
совпадение по нормализованному тексту запроса и по тексту вместе с категорией.
Это статистика аудита для M2, а не правило генерации кандидатов.

In [6]:
def normalize_query_text(series):
    return (
        series.astype("string")
        .fillna("<NA>")
        .str.lower()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

train_query_text = normalize_query_text(train_pairs.drop_duplicates("query_group")["search_query"])
benchmark_query_text = normalize_query_text(benchmark_queries["search_query"])
train_text_set = set(train_query_text)

train_text_category_set = set(
    zip(
        train_query_text.astype(str),
        train_pairs.drop_duplicates("query_group")["search_category"].astype(str),
    )
)
benchmark_text_category = list(
    zip(benchmark_query_text.astype(str), benchmark_queries["search_category"].astype(str))
)

benchmark_text_seen_in_train = benchmark_query_text.isin(train_text_set)
benchmark_text_category_seen_in_train = pd.Series(
    [pair in train_text_category_set for pair in benchmark_text_category],
    index=benchmark_queries.index,
)

history_overlap = pd.DataFrame(
    {
        "signal": [
            "full normalized search context",
            "normalized search_query text",
            "normalized search_query text + search_category",
        ],
        "benchmark_query_rows_seen_in_train": [
            len(exact_query_overlap),
            int(benchmark_text_seen_in_train.sum()),
            int(benchmark_text_category_seen_in_train.sum()),
        ],
        "benchmark_query_row_rate": [
            len(exact_query_overlap) / len(benchmark_queries),
            float(benchmark_text_seen_in_train.mean()),
            float(benchmark_text_category_seen_in_train.mean()),
        ],
    }
)
display(history_overlap)


,signal,benchmark_query_rows_seen_in_train,benchmark_query_row_rate
0,full normalized search context,108,0.044046
1,normalized search_query text,915,0.373165
2,normalized search_query text + search_category,891,0.363377


## 4. Текст и структура benchmark-корпуса

Этот блок смотрит только на searchable corpus: заполненность и длины полей,
дубликаты title и гранулярность категорий. Это определяет field ablations для
M1, но пока не выбирает веса и не обучает модель.


In [7]:
item_profile_columns = [
    "item_id",
    *ITEM_TEXT_COLUMNS,
    "item_category_id",
    "item_microcat_id",
    "item_location_id",
    "item_price",
    "item_rating",
    "item_rating_reviews_count",
    "item_is_phone_hidden",
    "item_is_message_forbidden",
]
benchmark_items = pd.read_parquet(BENCHMARK_ITEMS_PATH, columns=item_profile_columns)


def text_profile(frame, columns):
    rows = []
    for column in columns:
        values = frame[column].astype("string")
        lengths = values.fillna("").str.len()
        rows.append(
            {
                "field": column,
                "non_empty_rate": float(values.fillna("").str.strip().ne("").mean()),
                "median_chars": float(lengths.median()),
                "p95_chars": float(lengths.quantile(0.95)),
                "max_chars": int(lengths.max()),
            }
        )
    return pd.DataFrame(rows)

item_text_profile = text_profile(benchmark_items, ITEM_TEXT_COLUMNS)
display(item_text_profile)

normalized_titles = (
    benchmark_items["item_title_raw"]
    .astype("string")
    .fillna("")
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)
repeated_title_rows = int(normalized_titles.duplicated(keep=False).sum())

item_structure = pd.DataFrame(
    {
        "metric": [
            "unique benchmark item IDs",
            "unique item categories",
            "unique item microcategories",
            "unique item locations",
            "rows sharing a normalized title",
            "phone hidden rate",
            "message forbidden rate",
        ],
        "value": [
            benchmark_items["item_id"].nunique(),
            benchmark_items["item_category_id"].nunique(),
            benchmark_items["item_microcat_id"].nunique(),
            benchmark_items["item_location_id"].nunique(),
            repeated_title_rows,
            benchmark_items["item_is_phone_hidden"].mean(),
            benchmark_items["item_is_message_forbidden"].mean(),
        ],
    }
)
display(item_structure)


,field,non_empty_rate,median_chars,p95_chars,max_chars
0,item_title_raw,1.000000,35.0,50.0,100
1,item_infm_params_text,1.000000,738.0,2475.0,19068
2,item_description_raw,0.999815,1054.0,4205.0,8494


,metric,value
0,unique benchmark item IDs,189212.000000
1,unique item categories,47.000000
2,unique item microcategories,752.000000
3,unique item locations,2877.000000
4,rows sharing a normalized title,86253.000000
5,phone hidden rate,0.155947
6,message forbidden rate,0.026600


In [8]:
train_query_contexts = train_pairs.drop_duplicates("query_group")[[*SEARCH_COLUMNS, "query_group"]]
benchmark_query_text_profile = text_profile(
    benchmark_queries,
    ["search_query", "search_infm_params_text"],
).assign(split="benchmark")
train_query_text_profile = text_profile(
    train_query_contexts,
    ["search_query", "search_infm_params_text"],
).assign(split="train_unique_queries")

display(
    pd.concat([train_query_text_profile, benchmark_query_text_profile], ignore_index=True)
    [["split", "field", "non_empty_rate", "median_chars", "p95_chars", "max_chars"]]
)
print("Top benchmark search categories:")
display(
    benchmark_queries["search_category"]
    .value_counts(dropna=False)
    .head(10)
    .rename_axis("search_category")
    .reset_index(name="queries")
)


,split,field,non_empty_rate,median_chars,p95_chars,max_chars
0,train_unique_queries,search_query,1.000000,17.0,32.0,113
1,train_unique_queries,search_infm_params_text,0.647876,28.0,100.0,1079
2,benchmark,search_query,1.000000,22.0,39.0,70
3,benchmark,search_infm_params_text,0.369494,0.0,64.0,217


Top benchmark search categories:


,search_category,queries
0,114,2230
1,0,222


## 5. Frozen validation proxy

В следующих ячейках строится **proxy** для внутреннего сравнения экспериментов:
query groups с train-позитивами, чьи IDs присутствуют в `benchmark_items`,
ищутся в фиксированном benchmark corpus. Итоговая оценка всё равно будет
получена только после отправки `answer.csv`.

Split group-disjoint: один нормализованный search context не может попасть в
оба фолда, а все его proxy positives остаются на одной стороне.

In [9]:
def make_group_split(frame, group_column="query_group", test_size=0.20):
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=SEED)
    train_index, valid_index = next(splitter.split(frame, groups=frame[group_column]))
    train_part = frame.iloc[train_index].copy()
    valid_part = frame.iloc[valid_index].copy()
    assert set(train_part[group_column]).isdisjoint(set(valid_part[group_column]))
    return train_part, valid_part

proxy_train_pairs, proxy_valid_pairs = make_group_split(proxy_pairs)
proxy_split = pd.DataFrame(
    [
        {
            "partition": "train",
            "positive_pairs": len(proxy_train_pairs),
            "query_groups": proxy_train_pairs["query_group"].nunique(),
            "unique_positive_items": proxy_train_pairs["item_id"].nunique(),
        },
        {
            "partition": "validation",
            "positive_pairs": len(proxy_valid_pairs),
            "query_groups": proxy_valid_pairs["query_group"].nunique(),
            "unique_positive_items": proxy_valid_pairs["item_id"].nunique(),
        },
    ]
)
display(proxy_split)

VALIDATION_PROTOCOL = {
    "name": "benchmark_aligned_proxy_v1",
    "corpus": "benchmark_items.parquet",
    "labels": "train positives whose item_id exists in benchmark_items",
    "group": "all normalized search-side fields",
    "seed": SEED,
    "validation_query_group_fraction": 0.20,
}
VALIDATION_PROTOCOL


,partition,positive_pairs,query_groups,unique_positive_items
0,train,26377,21239,15569
1,validation,6633,5310,5200


{'name': 'benchmark_aligned_proxy_v1',
 'corpus': 'benchmark_items.parquet',
 'labels': 'train positives whose item_id exists in benchmark_items',
 'group': 'all normalized search-side fields',
 'seed': 42,
 'validation_query_group_fraction': 0.2}

## 6. Метрика и валидатор submission

Следующие ячейки реализуют macro Recall@50 по query groups и проверяют контракт
`answer.csv`: состав колонок, query IDs, item IDs, лимит в 50 кандидатов и
отсутствие повторов. Валидатор проверяет формат, но не может оценить benchmark
queries без их разметки.

In [10]:
def macro_recall_at_k(predictions, relevant, k=50):
    if not relevant:
        raise ValueError("At least one query with known relevant items is required.")
    recalls = []
    for query, gold_items in relevant.items():
        gold = set(gold_items)
        if not gold:
            raise ValueError(f"Query {query!r} has no gold items.")
        predicted = list(dict.fromkeys(predictions.get(query, [])))[:k]
        recalls.append(len(set(predicted) & gold) / len(gold))
    return float(np.mean(recalls))


def validate_answer_frame(answer, query_ids, corpus_ids):
    errors = []
    expected_columns = ["query_id", "answer"]
    if list(answer.columns) != expected_columns:
        errors.append(f"Expected columns {expected_columns}, got {list(answer.columns)}")
        return {"valid": False, "errors": errors}

    expected_query_ids = set(query_ids.astype(str))
    actual_query_ids = answer["query_id"].astype(str)
    if (actual_query_ids.str.len() != 16).any():
        errors.append("query_id must contain exactly 16 characters")
    if actual_query_ids.duplicated().any():
        errors.append("query_id contains duplicates")
    if set(actual_query_ids) != expected_query_ids:
        errors.append("query_id set differs from benchmark_queries")

    for row_number, raw_answer in enumerate(answer["answer"].fillna("").astype(str), start=2):
        item_ids = raw_answer.split()
        if raw_answer and raw_answer != " ".join(item_ids):
            errors.append(f"row {row_number}: answer must use single spaces without surrounding whitespace")
        if len(item_ids) > 50:
            errors.append(f"row {row_number}: contains {len(item_ids)} item IDs, expected at most 50")
        if len(item_ids) != len(set(item_ids)):
            errors.append(f"row {row_number}: duplicate item IDs")
        malformed = [item_id for item_id in item_ids if not HEX16.fullmatch(item_id)]
        if malformed:
            errors.append(f"row {row_number}: malformed item ID {malformed[0]!r}")
        unknown = [item_id for item_id in item_ids if item_id not in corpus_ids]
        if unknown:
            errors.append(f"row {row_number}: item ID absent from benchmark corpus {unknown[0]!r}")
    return {"valid": not errors, "errors": errors, "rows": len(answer)}

metric_sanity_relevant = {
    "A": {"a1"},
    "B": {"b1", "b2"},
    "C": {"c1"},
}
metric_sanity_predictions = {
    "A": ["a1"],
    "B": ["b1"],
    "C": [],
}
metric_sanity_value = macro_recall_at_k(metric_sanity_predictions, metric_sanity_relevant, k=50)
assert metric_sanity_value == 0.5, metric_sanity_value

empty_submission = pd.DataFrame({"query_id": benchmark_query_ids.astype(str), "answer": ""})
validation_smoke_test = validate_answer_frame(
    empty_submission,
    benchmark_query_ids,
    benchmark_item_id_set,
)
validation_smoke_test


{'valid': True, 'errors': [], 'rows': 2452}

In [11]:
M0_REPORT = {
    "train_rows": int(len(train_pairs)),
    "benchmark_queries": int(len(benchmark_queries)),
    "benchmark_items": int(len(benchmark_items)),
    "logical_train_queries": int(len(query_labels)),
    "median_known_positives_per_query": float(query_labels["n_known_positives"].median()),
    "benchmark_aligned_proxy_queries": int(proxy_labels.size),
    "exact_train_benchmark_query_context_overlap": int(len(exact_query_overlap)),
    "benchmark_query_text_rows_seen_in_train": int(benchmark_text_seen_in_train.sum()),
    "benchmark_query_text_category_rows_seen_in_train": int(benchmark_text_category_seen_in_train.sum()),
    "train_item_ids_in_benchmark_items": int(len(item_overlap)),
    "answer_validator_smoke_test": bool(validation_smoke_test["valid"]),
}

print("M0 summary")
for key, value in M0_REPORT.items():
    print(f"- {key}: {value}")
print("\nDecision")
print("M0 is complete only after reviewing these values and freezing benchmark_aligned_proxy_v1 or documenting why it is insufficient.")


M0 summary
- train_rows: 497673
- benchmark_queries: 2452
- benchmark_items: 189212
- logical_train_queries: 354313
- median_known_positives_per_query: 1.0
- benchmark_aligned_proxy_queries: 26549
- exact_train_benchmark_query_context_overlap: 108
- benchmark_query_text_rows_seen_in_train: 915
- benchmark_query_text_category_rows_seen_in_train: 891
- train_item_ids_in_benchmark_items: 18142
- answer_validator_smoke_test: True

Decision
M0 is complete only after reviewing these values and freezing benchmark_aligned_proxy_v1 or documenting why it is insufficient.


## Следующий этап

После просмотра итоговой сводки фиксируем `benchmark_aligned_proxy_v1` и
переходим к **E01 — plain BM25**. Если proxy оказался бы недостаточным, здесь
нужно было бы документировать ограничение и добавить train-corpus proxy.

ClearML намеренно не запускается в M0: это аудит, а не model experiment. Начиная
с E01 каждый измеренный retrieval run получает отдельную ClearML Task.